# 10 — Debugging & Profiling

**Yang akan lo pelajari:**
- Baca dan pahami traceback / error
- Debug dengan `%debug` dan `%pdb`
- Profiling performa: `%timeit`, `%prun`, `%lprun`
- Memory profiling
- Tips debugging efektif di Jupyter

---

## 1. Baca Traceback dengan Benar

In [ ]:
# Sengaja bikin error untuk lihat traceback
def proses_data(data):
    return [item['nilai'] for item in data]

def load_dan_proses(filepath):
    data = [{'nama': 'Budi', 'nilai': 85}, {'nama': 'Ani'}]  # Ani tidak punya 'nilai'!
    return proses_data(data)

try:
    hasil = load_dan_proses('data.csv')
except KeyError as e:
    print(f"Error: {type(e).__name__}: {e}")
    print("\nCara baca traceback:")
    print("1. Baca dari BAWAH ke atas")
    print("2. Baris paling bawah = error utama")
    print("3. Naik ke atas = call stack (siapa panggil siapa)")

In [ ]:
# Error types yang paling sering muncul
error_examples = [
    ("NameError",     "Variabel belum didefinisikan",    "x = undefined_var"),
    ("TypeError",     "Operasi pada tipe yang salah",    "'hello' + 5"),
    ("ValueError",    "Nilai tidak valid untuk operasi", "int('abc')"),
    ("IndexError",    "Index di luar range list",        "[1,2,3][10]"),
    ("KeyError",      "Key tidak ada di dict",           "{}.['key']"),
    ("AttributeError","Attribute/method tidak ada",      "None.upper()"),
    ("ImportError",   "Module tidak bisa diimport",      "import nonexistent"),
    ("ZeroDivisionError", "Dibagi nol",                  "1/0"),
    ("FileNotFoundError", "File tidak ditemukan",        "open('ghost.txt')"),
    ("IndentationError",  "Indentasi salah",             "Umumnya syntax error"),
]

print(f"{'Error Type':<20} {'Penyebab':<35} {'Contoh'}")
print("-" * 80)
for err, cause, ex in error_examples:
    print(f"{err:<20} {cause:<35} {ex}")

## 2. Debug dengan %debug

In [ ]:
# %debug — aktif setelah terjadi error
# Jalankan cell yang error, lalu di cell berikutnya ketik %debug
# Ini akan buka interactive debugger (pdb) di titik error

# Contoh:
def hitung_rata(data):
    total = sum(data)
    return total / len(data)   # error kalau data kosong

# Coba jalankan dengan data kosong
try:
    hitung_rata([])
except ZeroDivisionError:
    print("ZeroDivisionError terjadi!")
    print("Di situasi nyata, jalankan %debug di cell berikutnya")

In [ ]:
# Setelah %debug aktif, command pdb yang tersedia:
# n (next)     → execute baris berikutnya
# s (step)     → masuk ke dalam function
# c (continue) → lanjut sampai breakpoint berikutnya atau selesai
# q (quit)     → keluar dari debugger
# p var        → print nilai variabel
# l            → lihat kode di sekitar posisi saat ini
# u/d          → naik/turun di call stack

print("Command pdb yang perlu diingat:")
cmds = [
    ('n', 'next', 'Execute baris ini, lanjut ke berikutnya'),
    ('s', 'step', 'Masuk ke dalam function call'),
    ('c', 'continue', 'Lanjut sampai breakpoint/selesai'),
    ('q', 'quit', 'Keluar dari debugger'),
    ('p x', 'print', 'Print nilai variabel x'),
    ('l', 'list', 'Tampilkan kode sekitar posisi ini'),
    ('u', 'up', 'Naik ke frame di atas (pemanggil)'),
    ('d', 'down', 'Turun ke frame di bawah'),
]
for cmd, full, desc in cmds:
    print(f"  {cmd:<6} ({full:<8}) — {desc}")

In [ ]:
# %pdb on — aktifkan debugger otomatis setiap ada error
# %pdb off — matikan

# Alternatif: breakpoint() — built-in Python 3.7+
def fungsi_dengan_breakpoint(x):
    hasil = x * 2
    # breakpoint()   # ← uncomment untuk debug interaktif
    return hasil + 1

print(fungsi_dengan_breakpoint(5))

## 3. Teknik Debug Sederhana

In [ ]:
# Teknik 1: Print statements strategis
def pipeline(data):
    print(f"[DEBUG] Input: {data[:3]}... ({len(data)} items)")
    
    step1 = [x * 2 for x in data]
    print(f"[DEBUG] Setelah step1: {step1[:3]}")
    
    step2 = [x for x in step1 if x > 5]
    print(f"[DEBUG] Setelah filter: {len(step2)} items")
    
    return step2

hasil = pipeline([1, 2, 3, 4, 5, 6])
print(f"Hasil: {hasil}")

In [ ]:
# Teknik 2: assert untuk validasi asumsi
import numpy as np

def normalize(arr):
    assert isinstance(arr, np.ndarray), f"Input harus numpy array, bukan {type(arr)}"
    assert arr.ndim == 1, f"Input harus 1D, bukan {arr.ndim}D"
    assert len(arr) > 0, "Array tidak boleh kosong"
    
    min_val, max_val = arr.min(), arr.max()
    assert min_val != max_val, "Semua nilai sama, tidak bisa normalize"
    
    return (arr - min_val) / (max_val - min_val)

# Test normal
arr = np.array([1.0, 5.0, 3.0, 8.0, 2.0])
print(normalize(arr))

# Test error case
try:
    normalize([1, 2, 3])   # list, bukan numpy
except AssertionError as e:
    print(f"AssertionError: {e}")

In [ ]:
# Teknik 3: Isolasi masalah dengan binary search
# Kalau pipeline panjang error di tengah, coba:
# 1. Cek output step pertengahan
# 2. Kalau ok, cek step 3/4
# 3. Kalau tidak ok, cek step 1/4
# Seperti binary search — ini cara paling efisien

data = list(range(100))

def step_a(d): return [x**2 for x in d]
def step_b(d): return [x for x in d if x % 2 == 0]
def step_c(d): return sorted(d, reverse=True)[:10]
def step_d(d): return sum(d) / len(d)

# Debug per step
a = step_a(data)
print(f"A: {a[:5]}")

b = step_b(a)
print(f"B: {b[:5]}")

c = step_c(b)
print(f"C: {c}")

d = step_d(c)
print(f"D (final): {d}")

## 4. Profiling Performa

In [ ]:
import numpy as np

# %timeit — sudah kita bahas di notebook 04
# Recap cepat:

# Perbandingan: loop vs vectorized
data = list(range(100_000))
arr = np.array(data)

print("Python loop:")
%timeit sum(data)

print("\nNumPy vectorized:")
%timeit np.sum(arr)

In [ ]:
# %prun — profile seluruh function call (mana yang paling lambat)
def operasi_berat():
    data = list(range(10000))
    hasil = []
    for x in data:
        if x % 2 == 0:
            hasil.append(x ** 2)
    return sum(hasil)

# Jalankan profiler
%prun operasi_berat()

In [ ]:
# %%prun — profile seluruh cell
%%prun
import numpy as np

data = np.random.randn(10000)
hasil = np.sort(data)
mean = np.mean(hasil)
std = np.std(hasil)

In [ ]:
# Memory profiling dengan tracemalloc (built-in)
import tracemalloc

def operasi_memory():
    # Buat list besar
    big_list = list(range(1_000_000))
    # Convert ke set
    big_set = set(big_list)
    return len(big_set)

tracemalloc.start()
hasil = operasi_memory()
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Hasil: {hasil}")
print(f"Memory saat ini: {current / 1024 / 1024:.2f} MB")
print(f"Memory peak: {peak / 1024 / 1024:.2f} MB")

In [ ]:
# Bandingkan memory: list vs numpy vs generator
import sys

n = 1_000_000

python_list = list(range(n))
numpy_arr = np.arange(n)
gen = range(n)          # generator, tidak disimpan di memory

print(f"Python list  : {sys.getsizeof(python_list) / 1024 / 1024:.2f} MB")
print(f"NumPy array  : {numpy_arr.nbytes / 1024 / 1024:.2f} MB")
print(f"Range object : {sys.getsizeof(gen)} bytes (!!)")

## 5. Common Bugs dan Cara Fix-nya

In [ ]:
# BUG 1: Mutable default argument
# ❌ SALAH
def tambah_item_salah(item, lst=[]):
    lst.append(item)
    return lst

print(tambah_item_salah('a'))  # ['a']
print(tambah_item_salah('b'))  # ['a', 'b'] ← list SHARED across calls!
print(tambah_item_salah('c'))  # ['a', 'b', 'c']

print()

# ✅ BENAR
def tambah_item_benar(item, lst=None):
    if lst is None:
        lst = []
    lst.append(item)
    return lst

print(tambah_item_benar('a'))  # ['a']
print(tambah_item_benar('b'))  # ['b'] ← fresh list setiap call

In [ ]:
# BUG 2: Modifikasi list saat iterasi
angka = [1, 2, 3, 4, 5, 6]

# ❌ SALAH — skip elemen
for x in angka:
    if x % 2 == 0:
        angka.remove(x)
print("Salah:", angka)  # [1, 3, 5] — seharusnya [1, 3, 5]
                         # tapi bisa bug di kasus lain

# ✅ BENAR — buat list baru
angka = [1, 2, 3, 4, 5, 6]
angka_baru = [x for x in angka if x % 2 != 0]
print("Benar:", angka_baru)

In [ ]:
# BUG 3: Floating point precision
print(0.1 + 0.2)              # 0.30000000000000004
print(0.1 + 0.2 == 0.3)       # False!

# ✅ Fix
import math
print(math.isclose(0.1 + 0.2, 0.3))  # True
print(abs(0.1 + 0.2 - 0.3) < 1e-9)  # True (manual tolerance)

In [ ]:
# LATIHAN — Debug kode berikut
# Ada 3 bug di bawah ini, cari dan fix semuanya

def analisis_nilai(data):
    """
    Terima list nilai, return dict statistik.
    BUG: ada 3 error yang perlu difix
    """
    # Bug 1: tidak handle list kosong
    rata = sum(data) / len(data)
    
    # Bug 2: variable name typo
    maksimun = max(data)
    minimum = min(data)
    
    # Bug 3: return key salah
    return {
        'mean': rata,
        'max': maksimun,    # seharusnya 'maksimum'
        'min': minimum,
        'range': maksimun - minimum
    }

# Test
nilai = [85, 92, 78, 95, 88]
hasil = analisis_nilai(nilai)
print(hasil)

# Ini harusnya error — coba fix dulu
try:
    analisis_nilai([])  # list kosong
except ZeroDivisionError:
    print("\nFix Bug 1: tambah guard untuk list kosong")

---
## ➡️ Next
Satu notebook lagi! Lanjut ke **[11_best_practices.ipynb](11_best_practices.ipynb)** — cara bikin notebook yang rapi, reproducible, dan siap dipublikasikan.